[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/15_online_experimentation/15_proyecto_ab_interleaving.ipynb)

# Proyecto 15 · Simulador de A/B test + interleaving y análisis estadístico completo

**Nivel:** 🟠 Avanzado / 🔴 Experto · **Duración:** 4 h · **Hardware:** CPU (sin GPU) · **Unidades de Colab:** < 1
**Prerrequisitos:** lección 15. Conecta con el proyecto 14 (el dossier OPE decide qué va a A/B) y con 17 (monitoreo post-lanzamiento).

## 🎬 Contexto de negocio

Eres el/la *data scientist* de experimentación de **CineMatch**. El equipo de ranking tiene un **ranker B** (nueva versión del modelo de factorización, menos ruidoso) que quiere sustituir al **ranker A** de producción en la fila "Top 10 para ti". El VP de producto pregunta:

1. ¿Cuántos usuarios y cuántos días necesitamos para un A/B fiable?
2. ¿B es mejor? ¿Cuánto, con qué incertidumbre, y rompe algún guardrail?
3. ¿Podríamos haberlo sabido antes y con menos usuarios (CUPED, interleaving)?
4. Si el equipo mira el dashboard cada día, ¿nos engañaremos?

Como no tienes usuarios reales, construyes un **simulador**: una población de usuarios "gemelos" de MovieLens-100K (sus factores latentes y su actividad heterogénea vienen de los ratings reales), con un modelo de examen por posición. El simulador conoce los **resultados potenciales** de cada usuario con A y con B: puedes calificar tus estimadores contra la verdad, cosa imposible en producción.

## 📦 Dataset

**MovieLens-100K** (GroupLens; Harper & Konstan, 2015): 100.000 ratings de 943 usuarios sobre 1.682 películas. Se descarga de `files.grouplens.org` (si no hay red, el notebook genera ratings sintéticos con la misma forma para no bloquearse). De él extraemos factores latentes (SVD truncada) y la distribución de actividad, y con ellos simulamos `N_POP` usuarios.

## ✅ Entregables y rúbrica

| # | Entregable | Criterio |
|---|---|---|
| E1 | Power analysis con la varianza del periodo previo | n por grupo para MDE = 2 % (analítico y `statsmodels`, que deben coincidir ±1 %) y duración en días |
| E2 | Asignación por hash + SRM | p-valor SRM > 0,001 con tu asignación; detectar un SRM inyectado (p < 0,001) |
| E3 | Análisis del A/B | Lift relativo de B con IC 95 % (método delta para el ratio de medias); el IC **cubre** el lift real del simulador; guardrail "sesiones sin play" analizado |
| E4 | CUPED | Reducción de varianza ≥ 30 % y estimación compatible con la de E3 |
| E5 | Team Draft Interleaving | Implementación propia, test de preferencia sobre ≥ 5.000 usuarios con p < 0,01 a favor de B |
| E6 | Curvas de sensibilidad (bootstrap subsampling) | n para 80 % de potencia en A/B, A/B+CUPED e interleaving; interleaving necesita **≥ 5× menos** usuarios que el A/B |
| E7 | Peeking | En 200 A/A con análisis diario: FPR del test fijo con peeking > 10 % y del mSPRT ≤ 5 % |
| E8 | Memo de decisión | 6–10 líneas: decisión, efecto con IC, guardrails, riesgos (novelty, interferencia) |

In [ ]:
!pip install -q statsmodels scipy

In [ ]:
import warnings, time, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import NormalIndPower
warnings.filterwarnings("ignore")
seed = 42; rng = np.random.default_rng(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SCALE = "small"                                      # "small" ≈ 2-4 min en CPU | "full" más usuarios y repeticiones
N_POP = 100_000 if SCALE == "small" else 300_000     # usuarios elegibles durante las 2 semanas del experimento
N_BOOT = 150 if SCALE == "small" else 500            # repeticiones de bootstrap subsampling

def todo_guard(fn, *args, **kw):
    try:
        return fn(*args, **kw)
    except NotImplementedError:
        print(f"⏳ {fn.__name__}: TODO pendiente (solución de referencia al final)")
        return None

In [ ]:
# === Mundo simulado de CineMatch construido sobre MovieLens-100K ("gemelo digital") ===
import io, zipfile, urllib.request
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

def load_ml100k():
    """MovieLens-100K oficial de GroupLens; si no hay red, ratings sintéticos con la misma forma."""
    try:
        raw = urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            df = pd.read_csv(z.open("ml-100k/u.data"), sep="\t", names=["user", "item", "rating", "ts"])
        return df, "movielens-100k"
    except Exception as e:
        print(f"⚠️ No se pudo descargar MovieLens ({type(e).__name__}); uso ratings SINTÉTICOS con la misma forma.")
        g = np.random.default_rng(seed); nu, ni, k = 943, 1682, 8
        Pu, Qi = g.normal(0, 1, (nu, k)), g.normal(0, 1, (ni, k))
        pop_w = g.zipf(1.3, ni).clip(1, 600).astype(float); pop_w /= pop_w.sum()
        act = np.exp(g.normal(4.2, 0.9, nu)).astype(int).clip(20, 700)
        rows = []
        for u in range(nu):
            its = g.choice(ni, act[u], replace=False, p=pop_w)
            r = np.clip(np.round(3.5 + 1.2 * (Qi[its] @ Pu[u]) / np.sqrt(k) + g.normal(0, 0.8, len(its))), 1, 5)
            rows.append(pd.DataFrame(dict(user=u + 1, item=its + 1, rating=r, ts=0)))
        return pd.concat(rows, ignore_index=True), "synthetic"

ratings, DATA_SOURCE = load_ml100k()
n_u, n_i = ratings.user.max(), ratings.item.max()
centered = ratings.rating - ratings.groupby("user").rating.transform("mean")
R = csr_matrix((centered, (ratings.user - 1, ratings.item - 1)), shape=(n_u, n_i)).astype(float)
U_, s_, Vt_ = svds(R, k=16)                                   # factores latentes "reales" de MovieLens
P_ML, Q_ML = U_ * np.sqrt(s_), Vt_.T * np.sqrt(s_)
pop_all = np.bincount(ratings.item - 1, minlength=n_i)
acts_ml = np.bincount(ratings.user - 1, minlength=n_u)
print(f"Fuente: {DATA_SOURCE} | {len(ratings):,} ratings, {n_u} usuarios, {n_i} películas")

In [ ]:
TOPK, N_CAT = 10, 600
EXAM = (1 / np.log2(np.arange(2, TOPK + 2))).astype(np.float32)     # P(examinar posición k)
CATALOG = np.argsort(-pop_all)[:N_CAT]                              # las 600 películas más vistas
Qc = Q_ML[CATALOG].astype(np.float32)
lp = np.log1p(pop_all[CATALOG]); lp = ((lp - lp.mean()) / lp.std()).astype(np.float32)
AFF_SCALE = float((P_ML @ Q_ML[CATALOG].T).std())
PLAY_SCALE = 0.25

def topk(S):
    idx = np.argpartition(-S, TOPK, axis=1)[:, :TOPK]
    order = np.argsort(-np.take_along_axis(S, idx, 1), 1)
    return np.take_along_axis(idx, order, 1)

def build_population(n, seed=7, noise_a=1.0, noise_b=0.6):
    """Población de n usuarios 'gemelos' de usuarios de MovieLens (embedding + ruido; actividad
    heterogénea derivada del nº de ratings). Devuelve un dict con resultados potenciales:
      x_pre: plays en las 2 semanas PREVIAS (todos con el ranker A de producción)
      y_a / y_b: plays en las 2 semanas del experimento si el usuario estuviera en A / en B
      day: día (1-14) en que el usuario entra por primera vez en el experimento
      lists_a / lists_b / p_a / p_b: top-10 de cada ranker y probabilidad de play de cada ítem
    Ranker A = producción (MF + popularidad, ruidoso). Ranker B = candidato (mismo modelo, menos ruido)."""
    g = np.random.default_rng(seed)
    base = g.integers(0, n_u, n)
    emb = (P_ML[base] + g.normal(0, 0.5 * P_ML.std(), (n, P_ML.shape[1]))).astype(np.float32)
    la = np.log(acts_ml[base])
    sess_rate = np.exp(0.9 * (la - np.log(acts_ml).mean()) + g.normal(0, 0.5, n) + np.log(4))
    sess_pre, sess_exp = g.poisson(sess_rate), g.poisson(sess_rate)
    out = {k: np.zeros(n) for k in ["x_pre", "y_a", "y_b", "zero_a", "zero_b"]}
    out.update(lists_a=np.zeros((n, TOPK), np.int32), lists_b=np.zeros((n, TOPK), np.int32),
               p_a=np.zeros((n, TOPK), np.float32), p_b=np.zeros((n, TOPK), np.float32))
    for st in range(0, n, 20_000):
        sl = slice(st, st + 20_000); z = (emb[sl] @ Qc.T) / AFF_SCALE
        P = 1 / (1 + np.exp(-(1.2 * z + 0.3 * lp - 1.0)))                   # P(play | examinado) real
        A = topk(z + 0.6 * lp + noise_a * g.standard_normal(z.shape, dtype=np.float32))
        B = topk(z + 0.6 * lp + noise_b * g.standard_normal(z.shape, dtype=np.float32))
        rows = np.arange(z.shape[0])[:, None]
        out["p_a"][sl], out["p_b"][sl] = P[rows, A], P[rows, B]
        out["lists_a"][sl], out["lists_b"][sl] = A, B
        pa, pb = out["p_a"][sl] * EXAM * PLAY_SCALE, out["p_b"][sl] * EXAM * PLAY_SCALE
        out["x_pre"][sl] = g.binomial(sess_pre[sl][:, None], pa).sum(1)
        out["y_a"][sl] = g.binomial(sess_exp[sl][:, None], pa).sum(1)
        out["y_b"][sl] = g.binomial(sess_exp[sl][:, None], pb).sum(1)
        # guardrail: sesiones sin ningún play ("abandono de la home")
        p0a, p0b = np.prod(1 - pa, 1), np.prod(1 - pb, 1)
        out["zero_a"][sl] = g.binomial(sess_exp[sl], p0a) / np.maximum(sess_exp[sl], 1)
        out["zero_b"][sl] = g.binomial(sess_exp[sl], p0b) / np.maximum(sess_exp[sl], 1)
    out["sessions"] = sess_exp; out["day"] = g.integers(1, 15, n); out["user_id"] = np.arange(n) + 10_000_000
    return out

t0 = time.time()
pop_sim = build_population(N_POP)
print(f"Población simulada: {N_POP:,} usuarios en {time.time() - t0:.0f}s | plays medios A={pop_sim['y_a'].mean():.3f}, "
      f"B={pop_sim['y_b'].mean():.3f} (lift real oculto: {pop_sim['y_b'].mean() / pop_sim['y_a'].mean() - 1:+.2%})")

> 🔒 **Regla del juego:** en las partes de análisis solo puedes usar lo que verías en producción: para cada usuario, su grupo asignado, `x_pre` (plays previos), el resultado **del grupo en el que está** y su día de entrada. Las columnas `y_a` e `y_b` juntas (resultados potenciales) solo se usan para **calificar**. La celda siguiente crea esa vista "observable".

In [ ]:
def observe(pop, treated: np.ndarray) -> pd.DataFrame:
    """Lo que vería la plataforma: resultado solo del brazo asignado."""
    return pd.DataFrame({"user_id": pop["user_id"], "treated": treated.astype(int), "x_pre": pop["x_pre"],
                         "y": np.where(treated, pop["y_b"], pop["y_a"]),
                         "zero_rate": np.where(treated, pop["zero_b"], pop["zero_a"]), "day": pop["day"]})
TRUE_LIFT = pop_sim["y_b"].mean() / pop_sim["y_a"].mean() - 1      # ¡solo para calificar!

---
## Parte 1 · Diseño: power analysis (E1)

### TODO 1
Con los datos del **periodo previo** (`x_pre`, todos con A) estima μ y σ de "plays por usuario en 2 semanas". Calcula el **n por grupo** para detectar un MDE relativo del 2 % (α=0,05, potencia 0,8) con la fórmula y con `NormalIndPower`, y cuántos días harían falta si entran `N_POP/14` usuarios nuevos al día repartidos en dos grupos.

In [ ]:
def power_analysis(x_pre: np.ndarray, mde_rel: float = 0.02, alpha=0.05, power=0.8) -> dict:
    """Devuelve {'mu','sigma','n_formula','n_statsmodels','dias'}."""
    # TODO
    raise NotImplementedError
design = todo_guard(power_analysis, pop_sim["x_pre"])

## Parte 2 · Asignación y SRM (E2)

### TODO 2
1. `assign(user_ids, salt)`: hash MD5 de `f"{salt}:{user_id}"` → bucket 0–999 → tratamiento si bucket < 500.
2. `srm_pvalue(n_t, n_c)`: χ² de bondad de ajuste contra 50/50.
3. Simula un bug: el 3 % de los usuarios tratados no registra la exposición. ¿Lo detecta tu SRM?

In [ ]:
def assign(user_ids, salt: str) -> np.ndarray:
    # TODO
    raise NotImplementedError

def srm_pvalue(n_t: int, n_c: int) -> float:
    # TODO
    raise NotImplementedError

treated = todo_guard(assign, pop_sim["user_id"], "cinematch_top10_rankerB_v2")

## Parte 3 · Análisis del A/B (E3)

### TODO 3
- `ab_analysis(df)`: diferencia de medias con Welch, IC 95 %, p-valor y **lift relativo** $\bar Y_T/\bar Y_C - 1$ con IC por **método delta** (varianza del ratio de dos medias independientes: $\mathrm{Var}(\bar Y_T/\bar Y_C) \approx \frac{1}{\bar Y_C^2}\mathrm{Var}(\bar Y_T) + \frac{\bar Y_T^2}{\bar Y_C^4}\mathrm{Var}(\bar Y_C)$).
- Repite para el guardrail `zero_rate` (fracción de sesiones sin ningún play: **menos es mejor**).

In [ ]:
def ab_analysis(y_t: np.ndarray, y_c: np.ndarray) -> dict:
    """Devuelve {'diff','se','p','lift','lift_lo','lift_hi'}."""
    # TODO
    raise NotImplementedError

## Parte 4 · CUPED (E4)

### TODO 4
Implementa `cuped(y, x)` con θ estimado sobre todos los usuarios y repite el análisis de la Parte 3 con la métrica ajustada. Reporta la reducción de varianza $1 - \mathrm{Var}(Y^{cuped})/\mathrm{Var}(Y)$ y compárala con $\rho^2$.

In [ ]:
def cuped(y: np.ndarray, x: np.ndarray) -> tuple:
    """Devuelve (y_ajustada, theta)."""
    # TODO
    raise NotImplementedError

## Parte 5 · Team Draft Interleaving (E5)

### TODO 5
1. Implementa `team_draft(list_a, list_b, k, rng)` → `(lista, equipos)`.
2. `interleaving_deltas(pop, idx, rng)`: para cada usuario de `idx`, construye la lista mezclada de `lists_a`/`lists_b`, simula sus plays en `sessions` sesiones con `EXAM * p * PLAY_SCALE` (la `p` de cada ítem está en `p_a`/`p_b`) y devuelve $\Delta_i = \text{plays}_B - \text{plays}_A$.
3. Test t de una muestra sobre Δ con 5.000 usuarios **distintos** de los del A/B (aquí: una muestra aleatoria de la población).

In [ ]:
def team_draft(list_a, list_b, k, rng):
    # TODO
    raise NotImplementedError

def interleaving_deltas(pop, idx: np.ndarray, rng) -> np.ndarray:
    # TODO
    raise NotImplementedError

## Parte 6 · Sensibilidad: ¿cuántos usuarios necesita cada método? (E6)

### TODO 6
Con *bootstrap subsampling* (como el post de Netflix de 2017) estima la potencia para varios tamaños totales $n$: A/B (Welch), A/B + CUPED e interleaving (t de una muestra). Dibuja las tres curvas, calcula el $n$ para 80 % de potencia y el **ratio A/B / interleaving**.

Pista: para el A/B, toma $n/2$ usuarios al azar en tratamiento (usa `y_b`) y $n/2$ **distintos** en control (usa `y_a`).

In [ ]:
def sensitivity_curves(pop, deltas_pool: np.ndarray, n_grid, n_boot, rng) -> pd.DataFrame:
    """DataFrame con columnas n, power_ab, power_cuped, power_il."""
    # TODO
    raise NotImplementedError

## Parte 7 · El dashboard diario: peeking (E7)

### TODO 7
Simula **200 A/A tests** con la población (asignación aleatoria 50/50, ambos con A: usa `y_a`). Cada día $d=1..14$ analiza a los usuarios con `day ≤ d` (los que ya han entrado). Calcula la tasa de A/As en los que **algún día** p < 0,05 con el test fijo, y con p-valores **mSPRT** (mezcla normal, $\tau^2$ = (2 % de la media)²).

In [ ]:
def peeking_study(pop, n_aa: int, rng) -> dict:
    """Devuelve {'fpr_fixed_last_day','fpr_peeking','fpr_msprt'}."""
    # TODO
    raise NotImplementedError

## Parte 8 · Memo de decisión (E8)

### TODO 8 — escribe aquí tu memo
*Decisión (lanzar / no lanzar / iterar), efecto en plays con IC, guardrail, qué aportaron CUPED e interleaving, riesgos (novelty: solo 2 semanas; interferencia: ninguna aparente en este producto; efecto a largo plazo en retención: ¿qué surrogate usarías?).*

---
# ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa. Redefine cada función y ejecuta las comprobaciones de la rúbrica.

### Solución · Partes 1–2

In [ ]:
def power_analysis(x_pre, mde_rel=0.02, alpha=0.05, power=0.8) -> dict:
    mu, sigma = x_pre.mean(), x_pre.std(ddof=1)
    delta = mde_rel * mu
    n_formula = 2 * (stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)) ** 2 * sigma ** 2 / delta ** 2
    n_sm = NormalIndPower().solve_power(effect_size=delta / sigma, alpha=alpha, power=power, ratio=1.0)
    per_day_per_group = N_POP / 14 / 2
    return dict(mu=mu, sigma=sigma, n_formula=int(np.ceil(n_formula)), n_statsmodels=int(np.ceil(n_sm)),
                dias=int(np.ceil(n_formula / per_day_per_group)))

design = power_analysis(pop_sim["x_pre"])
print(design, "| E1", "✅" if abs(design["n_formula"] / design["n_statsmodels"] - 1) < 0.01 else "❌")
print(f"Con CV={design['sigma']/design['mu']:.2f}, MDE 2 % ⇒ {design['n_formula']:,} usuarios/grupo ≈ {design['dias']} días "
      "(redondea a semanas completas por la estacionalidad semanal).")

def assign(user_ids, salt):
    b = np.array([int(hashlib.md5(f"{salt}:{u}".encode()).hexdigest()[:8], 16) % 1000 for u in user_ids])
    return b < 500

def srm_pvalue(n_t, n_c):
    n = n_t + n_c
    return stats.chisquare([n_t, n_c], f_exp=[n / 2, n / 2]).pvalue

treated = assign(pop_sim["user_id"], "cinematch_top10_rankerB_v2")
p_ok = srm_pvalue(treated.sum(), (~treated).sum())
lost = treated & (np.random.default_rng(1).random(N_POP) < 0.03)
p_bug = srm_pvalue((treated & ~lost).sum(), (~treated).sum())
print(f"SRM limpio p={p_ok:.3f} | con bug p={p_bug:.1e}  →  E2 {'✅' if (p_ok > 1e-3 and p_bug < 1e-3) else '❌'}")
obs = observe(pop_sim, treated)

### Solución · Partes 3–4

In [ ]:
def ab_analysis(y_t, y_c) -> dict:
    mt, mc = y_t.mean(), y_c.mean()
    vt, vc = y_t.var(ddof=1) / len(y_t), y_c.var(ddof=1) / len(y_c)
    se = np.sqrt(vt + vc); p = stats.ttest_ind(y_t, y_c, equal_var=False).pvalue
    lift = mt / mc - 1
    se_lift = np.sqrt(vt / mc ** 2 + mt ** 2 * vc / mc ** 4)              # método delta
    return dict(diff=mt - mc, se=se, p=p, lift=lift, lift_lo=lift - 1.96 * se_lift, lift_hi=lift + 1.96 * se_lift)

def cuped(y, x):
    theta = np.cov(y, x)[0, 1] / np.var(x, ddof=1)
    return y - theta * (x - x.mean()), theta

T = obs.treated.values == 1
res = ab_analysis(obs.y.values[T], obs.y.values[~T])
grd = ab_analysis(obs.zero_rate.values[T], obs.zero_rate.values[~T])
y_cu, theta = cuped(obs.y.values, obs.x_pre.values)
# Para el lift relativo con CUPED usamos como denominador la media de control sin ajustar:
lift_cu = (y_cu[T].mean() - y_cu[~T].mean()) / obs.y.values[~T].mean()
se_cu = np.sqrt(y_cu[T].var(ddof=1) / T.sum() + y_cu[~T].var(ddof=1) / (~T).sum()) / obs.y.values[~T].mean()
var_red = 1 - y_cu.var() / obs.y.values.var(); rho2 = np.corrcoef(obs.y, obs.x_pre)[0, 1] ** 2

tab = pd.DataFrame({"A/B": [res["lift"], res["lift_lo"], res["lift_hi"], res["p"]],
                    "A/B + CUPED": [lift_cu, lift_cu - 1.96 * se_cu, lift_cu + 1.96 * se_cu, 2 * stats.norm.sf(abs(lift_cu / se_cu))],
                    "Guardrail: sesiones sin play": [grd["lift"], grd["lift_lo"], grd["lift_hi"], grd["p"]]},
                   index=["lift", "IC95 inf", "IC95 sup", "p"]).T
display(tab.round(4)); print(f"Lift REAL del simulador: {TRUE_LIFT:+.2%}")
print("E3", "✅" if res["lift_lo"] <= TRUE_LIFT <= res["lift_hi"] else "❌",
      f"| E4 reducción de varianza {var_red:.0%} (ρ²={rho2:.0%}) θ={theta:.2f}", "✅" if var_red >= 0.30 else "❌")

### Solución · Parte 5

In [ ]:
def team_draft(list_a, list_b, k, rng):
    out, team, used = [], [], set(); ia = ib = na = nb = 0
    while len(out) < k:
        pick_a = (na < nb) or (na == nb and rng.random() < 0.5)
        for attempt in range(2):
            src, i = (list_a, ia) if pick_a else (list_b, ib)
            while i < len(src) and src[i] in used: i += 1
            if i < len(src): break
            pick_a = not pick_a                                     # ese ranker se quedó sin ítems
        else:
            break
        out.append(src[i]); used.add(src[i]); team.append(0 if pick_a else 1)
        if pick_a: ia, na = i + 1, na + 1
        else: ib, nb = i + 1, nb + 1
    return out, team

def interleaving_deltas(pop, idx, rng):
    d = np.empty(len(idx))
    for j, u in enumerate(idx):
        la, lb = pop["lists_a"][u].tolist(), pop["lists_b"][u].tolist()
        lst, team = team_draft(la, lb, TOPK, rng)
        pmap = dict(zip(la, pop["p_a"][u])); pmap.update(zip(lb, pop["p_b"][u]))
        p = np.array([pmap[i] for i in lst]) * EXAM[:len(lst)] * PLAY_SCALE
        plays = rng.binomial(pop["sessions"][u], p); team = np.array(team)
        d[j] = plays[team == 1].sum() - plays[team == 0].sum()
    return d

g_il = np.random.default_rng(3)
il_idx = g_il.choice(N_POP, 5000, replace=False)
d5k = interleaving_deltas(pop_sim, il_idx, g_il)
t_il = stats.ttest_1samp(d5k, 0)
print(f"Interleaving (5.000 usuarios): Δ medio={d5k.mean():+.4f}, usuarios con preferencia≠0 = {(d5k != 0).mean():.0%},"
      f" p={t_il.pvalue:.1e}  →  E5 {'✅' if (t_il.pvalue < 0.01 and d5k.mean() > 0) else '❌'}")

### Solución · Parte 6

In [ ]:
def sensitivity_curves(pop, deltas_pool, n_grid, n_boot, rng) -> pd.DataFrame:
    ya, yb, x = pop["y_a"], pop["y_b"], pop["x_pre"]
    theta = np.cov(ya, x)[0, 1] / np.var(x, ddof=1); xm = x.mean()
    rows = []
    for n in n_grid:
        h = n // 2; hits = np.zeros(3)
        for _ in range(n_boot):
            it, ic = rng.choice(len(ya), h), rng.choice(len(ya), h)
            r = stats.ttest_ind(yb[it], ya[ic], equal_var=False)
            hits[0] += (r.pvalue < 0.05) and (r.statistic > 0)
            rc = stats.ttest_ind(yb[it] - theta * (x[it] - xm), ya[ic] - theta * (x[ic] - xm), equal_var=False)
            hits[1] += (rc.pvalue < 0.05) and (rc.statistic > 0)
            d = deltas_pool[rng.choice(len(deltas_pool), n)]
            ri = stats.ttest_1samp(d, 0)
            hits[2] += (ri.pvalue < 0.05) and (ri.statistic > 0)
        rows.append(dict(n=n, power_ab=hits[0] / n_boot, power_cuped=hits[1] / n_boot, power_il=hits[2] / n_boot))
    return pd.DataFrame(rows)

t0 = time.time(); g_s = np.random.default_rng(4)
pool_idx = g_s.choice(N_POP, 20_000, replace=False)
deltas_pool = interleaving_deltas(pop_sim, pool_idx, g_s)
n_grid = np.unique(np.logspace(2, np.log10(N_POP), 12).astype(int))
curves = sensitivity_curves(pop_sim, deltas_pool, n_grid, N_BOOT, g_s)
print(f"{time.time() - t0:.0f}s")
def n80(col):
    ok = curves[col] >= 0.8
    return int(curves.n[ok].iloc[0]) if ok.any() else np.inf
n_ab, n_cu, n_il = n80("power_ab"), n80("power_cuped"), n80("power_il")
plt.figure(figsize=(9, 4.3))
for col, lab, c in [("power_ab", "A/B", "#0072b2"), ("power_cuped", "A/B + CUPED", "#009e73"), ("power_il", "Interleaving (Team Draft)", "#d55e00")]:
    plt.semilogx(curves.n, curves[col], "o-", lw=2, label=lab, color=c)
plt.axhline(0.8, ls="--", color="k"); plt.xlabel("usuarios totales en el experimento (log)"); plt.ylabel("potencia")
plt.title("Sensibilidad por bootstrap subsampling (ranker B vs A)"); plt.legend(); plt.show()
print(f"n para 80 % de potencia → A/B: {n_ab:,} | A/B+CUPED: {n_cu:,} | interleaving: {n_il:,}  ⇒ ratio A/B/IL ≈ {n_ab / n_il:.0f}×"
      f"  →  E6 {'✅' if n_ab / n_il >= 5 else '❌'}")

### Solución · Parte 7

In [ ]:
def peeking_study(pop, n_aa, rng) -> dict:
    y, day = pop["y_a"], pop["day"]
    tau2 = (0.02 * y.mean()) ** 2
    any_fixed, any_msprt, last_fixed = 0, 0, 0
    for _ in range(n_aa):
        grp = rng.random(len(y)) < 0.5
        p_days, diffs, vars_ = [], [], []
        for d in range(1, 15):
            m = day <= d; yt, yc = y[m & grp], y[m & ~grp]
            diff = yt.mean() - yc.mean(); v = yt.var(ddof=1) / len(yt) + yc.var(ddof=1) / len(yc)
            diffs.append(diff); vars_.append(v); p_days.append(2 * stats.norm.sf(abs(diff) / np.sqrt(v)))
        V, D = np.array(vars_), np.array(diffs)
        lam = np.sqrt(V / (V + tau2)) * np.exp(tau2 * D ** 2 / (2 * V * (V + tau2)))
        p_av = np.minimum.accumulate(np.minimum(1, 1 / lam))
        any_fixed += min(p_days) < 0.05; last_fixed += p_days[-1] < 0.05; any_msprt += p_av.min() < 0.05
    return dict(fpr_fixed_last_day=last_fixed / n_aa, fpr_peeking=any_fixed / n_aa, fpr_msprt=any_msprt / n_aa)

t0 = time.time(); peek = peeking_study(pop_sim, 200, np.random.default_rng(5)); print(f"{time.time() - t0:.0f}s", peek)
plt.figure(figsize=(6.5, 3.6))
plt.bar(["Test fijo\n(solo día 14)", "Test fijo\n+ peeking diario", "mSPRT\n+ peeking diario"],
        [peek["fpr_fixed_last_day"] * 100, peek["fpr_peeking"] * 100, peek["fpr_msprt"] * 100], color=["#9e9e9e", "#c53030", "#009e73"])
plt.axhline(5, ls="--", color="k"); plt.ylabel("falsos positivos en A/A (%)"); plt.title("El coste de mirar el dashboard cada día"); plt.show()
print("E7", "✅" if (peek["fpr_peeking"] > 0.10 and peek["fpr_msprt"] <= 0.05) else "❌")

In [ ]:
# Resumen visual del experimento: efecto acumulado día a día (lo que vería el dashboard) con IC fijo e IC "siempre válido"
days_, est_, lo_, hi_ = [], [], [], []
for d in range(1, 15):
    m = obs.day.values <= d
    r = ab_analysis(obs.y.values[m & T], obs.y.values[m & ~T]); days_.append(d); est_.append(r["lift"]); lo_.append(r["lift_lo"]); hi_.append(r["lift_hi"])
plt.figure(figsize=(8, 3.8))
plt.fill_between(days_, np.array(lo_) * 100, np.array(hi_) * 100, alpha=0.25, color="#0072b2", label="IC 95 % (horizonte fijo)")
plt.plot(days_, np.array(est_) * 100, "o-", color="#0072b2", label="lift estimado acumulado")
plt.axhline(TRUE_LIFT * 100, color="k", ls="--", label="lift real (simulador)"); plt.axhline(0, color="k", lw=0.5)
plt.xlabel("día del experimento"); plt.ylabel("lift en plays (%)"); plt.title("Evolución del A/B ranker B vs A"); plt.legend(fontsize=8); plt.show()

### Solución · Memo de decisión (ejemplo; ajusta las cifras a tu ejecución)

> **Decisión: lanzar el ranker B en la fila "Top 10 para ti".** El A/B con N_POP usuarios durante 14 días muestra un aumento de plays por usuario del orden de +4–5 % (IC 95 % excluye el 0; con CUPED el IC es ~2× más estrecho porque la varianza baja >50 %). El guardrail "sesiones sin play" mejora (baja), no empeora. Interleaving habría detectado la preferencia por B con un orden de magnitud menos usuarios: para la próxima iteración del ranker, usaremos interleaving como primera fase de poda y reservaremos el A/B para medir el impacto en negocio. El dashboard diario se analizará con p-valores siempre válidos (mSPRT) para evitar decisiones precipitadas. **Riesgos:** 2 semanas no permiten descartar novelty (mirar el efecto por cohorte de entrada); el efecto en retención a 3 meses no se observa: mantener un *holdout* del 1 % con A durante 8 semanas y estimar el efecto con un surrogate index validado (horas, días activos, títulos distintos).

---
## 🚀 Retos extra (nivel experto)

1. **Novelty:** añade al simulador un efecto que decae ($\tau_d = \tau_\infty + (\tau_0 - \tau_\infty)e^{-d/\kappa}$) y detecta el decaimiento comparando cohortes por día de entrada.
2. **Group sequential:** implementa fronteras O'Brien–Fleming con *alpha spending* de Lan–DeMets para 2 análisis intermedios + el final, y compara potencia con mSPRT.
3. **Multileaving:** compara 4 rankers a la vez con *Team Draft Multileaving* (Schuth et al.) y estima cuántos usuarios necesitas para ordenar los 4.
4. **Interferencia:** introduce un catálogo "de estreno" con licencias limitadas (cada título solo puede reproducirse N veces al día) y mide el sesgo del A/B por usuario frente a un *switchback* por días.
5. **HTE:** estima el efecto de B por cuartil de actividad con un T-learner y con `econml` (`CausalForestDML`); ¿a quién ayuda más?
6. **Herramientas:** reproduce el análisis con GrowthBook (open source) apuntando a un DuckDB con las tablas `exposures` y `events` que generas desde el simulador.

## 🤔 Reflexión (producción / MLOps)

- ¿Dónde vive la lógica de asignación (hash + salt) en vuestra arquitectura, y cómo garantizas que el servicio de recomendación y el pipeline de análisis usan **la misma**?
- ¿Cómo versionarías la definición de métricas (SQL) para que un experimento de hoy sea comparable con uno de hace un año?
- ¿Qué alertas automáticas pondrías (SRM, guardrails, A/A continuos) y quién recibe la alerta?
- Tras lanzar B, ¿cómo conectas el resultado del A/B con el monitoreo del módulo 17 (deriva de las métricas online frente a las del experimento)?